# Sentiment Analysis using NLP Pipeline and Machine Learning

This project demonstrates the complete NLP pipeline including preprocessing,
feature extraction using TF-IDF, and sentiment classification using multiple ML models.

## 📂 Data Loading

In this step, we load the dataset containing text reviews and sentiment labels.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving archive (3).zip to archive (3).zip


In [15]:
import zipfile

with zipfile.ZipFile("archive (3).zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [16]:
import os

os.listdir("data")

['Valid.csv', 'Train.csv', 'Test.csv']

In [ ]:
import pandas as pd

df = pd.read_csv("data/Train.csv")
df.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [17]:
df.columns


Index(['text', 'label', 'cleaned'], dtype='object')

In [20]:
df

,text,label,cleaned
0,I grew up (b. 1965) watching and loving the Th...,0,grew b watch love thunderbird mate school watc...
1,"When I put this movie in my DVD player, and sa...",0,put movi dvd player sat coke chip expect hope ...
2,Why do people who do not know what a particula...,0,peopl know particular time past like feel need...
3,Even though I have great interest in Biblical ...,0,even though great interest biblic movi bore de...
4,Im a die hard Dads Army fan and nothing will e...,1,im die hard dad armi fan noth ever chang got t...
...,...,...,...
39995,"""Western Union"" is something of a forgotten cl...",1,western union someth forgotten classic western...
39996,This movie is an incredible piece of work. It ...,1,movi incred piec work explor everi nook cranni...
39997,My wife and I watched this movie because we pl...,0,wife watch movi plan visit sicili stromboli so...
39998,"When I first watched Flatliners, I was amazed....",1,first watch flatlin amaz necessari featur good...


## 🧹 Data Preprocessing

In this step, we clean the text data by:
- Converting to lowercase
- Removing punctuation and special characters
- Removing stopwords
- Applying stemming

### Preprocessing Function

In [40]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Function to clean and preprocess text data
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Applying preprocessing to the dataset

### Apply Preprocessing

In [19]:
df['cleaned'] = df['text'].apply(preprocess)
df[['text','cleaned']].head()

,text,cleaned
0,I grew up (b. 1965) watching and loving the Th...,grew b watch love thunderbird mate school watc...
1,"When I put this movie in my DVD player, and sa...",put movi dvd player sat coke chip expect hope ...
2,Why do people who do not know what a particula...,peopl know particular time past like feel need...
3,Even though I have great interest in Biblical ...,even though great interest biblic movi bore de...
4,Im a die hard Dads Army fan and nothing will e...,im die hard dad armi fan noth ever chang got t...


## 🔢 Feature Engineering

We use TF-IDF vectorization to convert text into numerical features that machine learning models can understand.

### Features (TF-IDF BEST)

In [41]:
# Converting text data into numerical features using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned'])
y = df['label']

1. Check Shape

In [29]:
print(X.shape)

(40000, 5000)


# Reduced features to 5000 to improve model efficiency and avoid overfitting

2. View Sample Data

In [30]:
print(X[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 63 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 1959)	0.1505085905552037
  (0, 4846)	0.10535843511495116
  (0, 2665)	0.12809431099144974
  (0, 2758)	0.1587516805059731
  (0, 3868)	0.30973008133616164
  (0, 3321)	0.13023999913619735
  (0, 2682)	0.1819740826520132
  (0, 4830)	0.1369891897593505
  (0, 3879)	0.13908675972270693
  (0, 3117)	0.08154267081832854
  (0, 102)	0.15125630111732127
  (0, 988)	0.1334733423518632
  (0, 376)	0.12393259425473047
  (0, 233)	0.10597409140712923
  (0, 1752)	0.11530863453736845
  (0, 4545)	0.1102216821243848
  (0, 741)	0.10873336239886537
  (0, 3907)	0.052071826356090783
  (0, 2929)	0.03772644058693511
  (0, 2139)	0.09036355282699747
  (0, 4958)	0.05594206415468972
  (0, 1871)	0.053363396812847796
  (0, 1892)	0.1527382783453851
  (0, 738)	0.1121744831447756
  (0, 1248)	0.09728036429420016
  :	:
  (0, 705)	0.12888168547481912
  (0, 4248)	0.07234752233752607
  (0, 3670)	0.18562793044

3. Convert to Array

In [31]:
X.toarray()[0]

array([0., 0., 0., ..., 0., 0., 0.])

4. View Feature Names

In [32]:
tfidf.get_feature_names_out()[:20]

array(['aaron', 'abandon', 'abc', 'abduct', 'abil', 'abl', 'aboard',
       'abomin', 'abort', 'abound', 'abraham', 'abrupt', 'abruptli',
       'absenc', 'absent', 'absolut', 'absorb', 'abstract', 'absurd',
       'abund'], dtype=object)

## 🔀 Train-Test Split

Splitting the dataset into training and testing sets (80% training, 20% testing).

### Train-Test Split

In [42]:
# Splitting data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
print(X_train.shape)
print(X_test.shape)

(32000, 5000)
(8000, 5000)


In [35]:
print(y_train.shape)
print(y_test.shape)

(32000,)
(8000,)


In [37]:
# Checking data split
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (32000, 5000)
Test shape: (8000, 5000)


## 🤖 Model Training

Training multiple machine learning models:
- Logistic Regression
- Naive Bayes
- Decision Tree

### Train Models

In [38]:
# Training machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

lr = LogisticRegression()
nb = MultinomialNB()
dt = DecisionTreeClassifier()

lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
dt.fit(X_train, y_train)

DecisionTreeClassifier()

## 📊 Model Evaluation

Evaluating model performance using:
- Accuracy
- Precision
- Recall
- F1-score

### Evaluation

In [39]:
# Evaluating model performance
from sklearn.metrics import accuracy_score, classification_report

models = {"Logistic": lr, "Naive Bayes": nb, "Decision Tree": dt}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


Logistic
Accuracy: 0.884375
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      3966
           1       0.88      0.89      0.89      4034

    accuracy                           0.88      8000
   macro avg       0.88      0.88      0.88      8000
weighted avg       0.88      0.88      0.88      8000


Naive Bayes
Accuracy: 0.849625
              precision    recall  f1-score   support

           0       0.85      0.84      0.85      3966
           1       0.85      0.85      0.85      4034

    accuracy                           0.85      8000
   macro avg       0.85      0.85      0.85      8000
weighted avg       0.85      0.85      0.85      8000


Decision Tree
Accuracy: 0.719
              precision    recall  f1-score   support

           0       0.72      0.72      0.72      3966
           1       0.72      0.72      0.72      4034

    accuracy                           0.72      8000
   macro avg       0.72      0.72   

 ### 🧠 Insights


- Logistic Regression achieved the highest accuracy (88.4%) and performed best for this dataset.
- Naive Bayes also performed well but assumes feature independence.
- Decision Tree showed lower performance due to overfitting and handling high-dimensional data poorly.
- TF-IDF improved model performance by assigning importance to meaningful words.
- Preprocessing helped reduce noise and improve overall accuracy.